In [1]:
import os
import csv
import json
import shutil
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import kagglehub

from pyAudioAnalysis import audioTrainTest as aT
from pyAudioAnalysis import MidTermFeatures as mtf

from sklearn.metrics import classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

/home/Cerviniadmin/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/Cerviniadmin/.local/lib/python3.10/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [2]:
# np.Inf was removed in the NumPy 2.0 release. Use np.inf instead.
if not hasattr(np, "Inf"):
    np.Inf = np.inf
if not hasattr(np, "NaN"):
    np.NaN = np.nan

In [3]:
dataset_path = Path(kagglehub.dataset_download(
    "tommyngx/fluent-speech-corpus"
))
print("dataset_path:", dataset_path)

dataset_path: /home/Cerviniadmin/.cache/kagglehub/datasets/tommyngx/fluent-speech-corpus/versions/1


In [5]:
def load_complete_map(dataset_path):
    """
    Carica i metadati del Fluent Speech Commands dataset, fondendo TUTTI
    gli split ufficiali (train/valid/test forniti da Kaggle) in un unico
    pool di campioni. Lo split finale (train/dev/test) con i vincoli
    richiesti (eta/genere/speaker disjoint) viene ricostruito a parte
    da `build_speaker_splits`, quindi qui non importa da quale csv
    ufficiale provenga ciascuna riga.

    Per ogni campione vengono raccolti:
        - rel_path:    percorso relativo del file .wav
        - speaker_id:  id del parlante
        - gender:      "M" o "F"
        - age_range:   fascia di eta originale (es. "22-40", "41-65", "65+")
        - action, object, location: i 3 slot semantici dell'intento (FSC)
    """
    csv_demographics = list(Path(dataset_path).rglob("speaker_demographics.csv"))[0]

    gender_map = {}
    age_map = {}
    with open(csv_demographics, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            sid = row["speakerId"].strip()
            g = row["gender"].strip().lower()
            age_range = row["ageRange"].strip()

            if g.startswith("f"):
                gender_map[sid] = "F"
            elif g.startswith("m"):
                gender_map[sid] = "M"

            age_map[sid] = age_range

    audio_metadata = []

    # Tutti i csv per-utterance tranne quello demografico: includono sia
    # i csv ufficiali di train, sia quelli di valid/test (qualunque sia il
    # loro nome, es. train_data.csv / valid_data.csv / test_data.csv).
    # Li fondiamo tutti: lo split definitivo lo facciamo noi dopo.
    csv_files = [p for p in Path(dataset_path).rglob("*.csv")
                 if "demographics" not in p.name]

    seen_rows = set()

    for csv_file in csv_files:
        with open(csv_file, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)

            for row in reader:
                sid = row.get("speakerId", "").strip()
                rel_path = row.get("path", "").strip()

                action = row.get("action", "unknown").strip().lower()
                obj = row.get("object", "unknown").strip().lower()
                location = row.get("location", "unknown").strip().lower()

                gender = gender_map.get(sid)
                age_range = age_map.get(sid)

                if not (gender and rel_path):
                    continue

                # Evita duplicati nel caso lo stesso file compaia in piu'
                # di un csv (non dovrebbe succedere con gli split ufficiali,
                # ma e' una protezione a basso costo dato che li fondiamo).
                dedup_key = (sid, rel_path)
                if dedup_key in seen_rows:
                    continue
                seen_rows.add(dedup_key)

                audio_metadata.append({
                    "rel_path": rel_path,
                    "speaker_id": sid,
                    "gender": gender,
                    "age_range": age_range,
                    "action": action,
                    "object": obj,
                    "location": location,
                })

    print(f"Loaded metadata: {len(audio_metadata)} samples")
    return audio_metadata


In [6]:
def resolve_audio_path(dataset_path, rel_path):
    dataset_path = Path(dataset_path)

    # Extract the nested top-level directory name dynamically if needed,
    # or explicitly add the known folder to the search tree
    nested_dir = dataset_path / "fluent_speech_commands_dataset"

    candidates = [
        dataset_path / rel_path,
        nested_dir / rel_path,  # <-- Added this critical candidate
        dataset_path / Path(rel_path).name,
        nested_dir / Path(rel_path).name,
        dataset_path / "data" / rel_path,
        dataset_path / "wav" / rel_path,
        dataset_path / "audio" / rel_path,
        Path(rel_path)
    ]

    for c in candidates:
        if c.exists():
            return c

    return None

In [7]:
audio = load_complete_map(dataset_path)
print(audio[0]["rel_path"])
print({k: audio[0][k] for k in ("speaker_id", "gender", "age_range", "action", "object", "location")})


Loaded metadata: 30043 samples
wavs/speakers/2BqVo8kVB2Skwgyb/0a3129c0-4474-11e9-a9a5-5dbec3b8816a.wav
{'speaker_id': '2BqVo8kVB2Skwgyb', 'gender': 'F', 'age_range': '22-40', 'action': 'change language', 'object': 'none', 'location': 'none'}


In [8]:
SLOTS = ["action", "object", "location"]


def build_speaker_splits(audio_metadata):
    """
    Costruisce gli insiemi di speaker_id per train e test secondo i
    vincoli richiesti:

      - train contiene SOLO uomini con fascia di eta "22-40"
      - test contiene tutti gli altri speaker (qualsiasi genere/eta)
      - i parlanti di train non compaiono MAI in test e viceversa

    Ritorna 2 set di speaker_id: (train_speakers, test_speakers)
    """
    speakers = {}
    for m in audio_metadata:
        speakers[m["speaker_id"]] = (m["gender"], m["age_range"])

    # Seleziona chi va in train (Maschi, 22-40)
    eligible = sorted([sid for sid, (g, a) in speakers.items()
                        if g == "M" and a == "22-40"])

    # Tutti gli altri vanno in test
    rest = sorted([sid for sid in speakers if sid not in eligible])
    young_female = set()
    middle_age_female = set()
    old_female = set()
    middle_age_male = set()
    old_male = set()
    for sid in rest:
        g, a = speakers[sid]
        if g == "F" and a == "22-40":
            young_female.add(sid)
        if g == "F" and a == "41-65":
            middle_age_female.add(sid)
        if g == "F" and a == "65+":
            old_female.add(sid)
        if g == "M" and a == "41-65":
            middle_age_male.add(sid)
        if g == "M" and a == "65+":
            old_male.add(sid)

    if len(eligible) == 0:
        raise ValueError(
            "Nessuno speaker soddisfa il vincolo richiesto per train "
            "(gender == 'M' e age_range == '22-40'). Verifica i valori "
            "reali della colonna ageRange nel dataset."
        )

    train_speakers = set(eligible)
    test_speakers = set(rest)


    print(f"Speaker totali: {len(speakers)}")
    print(f"  -> train (M, 22-40): {len(train_speakers)} speaker")
    print(f"  -> test: donne giovani: {len(young_female)} speaker")
    print(f"  -> test: donne a 41-65: {len(middle_age_female)} speaker")
    print(f"  -> test: donne a 65+: {len(old_female)} speaker")
    print(f"  -> test: uomini a 41-65: {len(middle_age_male)} speaker")
    print(f"  -> test: uomini a 65+: {len(old_male)} speaker")

    # Garanzia esplicita di disgiunzione
    assert train_speakers.isdisjoint(test_speakers)
    assert young_female.isdisjoint(middle_age_female)
    assert young_female.isdisjoint(old_female)
    assert middle_age_female.isdisjoint(old_female)
    assert middle_age_female.isdisjoint(old_male)
    assert old_female.isdisjoint(middle_age_male)
    assert old_female.isdisjoint(old_male)
    assert old_male.isdisjoint(middle_age_male)
    assert middle_age_male.isdisjoint(old_male)
    assert len(young_female) + len(middle_age_female) + len(old_female) + len(middle_age_male) + len(old_male) == len(test_speakers)

    return train_speakers, young_female, middle_age_female, old_female, middle_age_male, old_male


def build_split_workspace(audio_metadata, dataset_path, speaker_ids, workspace_dir):
    """
    Copia i file audio degli speaker indicati dentro `workspace_dir`,
    organizzati in sotto-cartelle per classe per OGNUNO dei 3 slot
    (action / object / location), cosi' che pyAudioAnalysis possa
    allenare un classificatore per slot (1 cartella = 1 classe).

    workspace_dir/
        action/<classe_action>/*.wav
        object/<classe_object>/*.wav
        location/<classe_location>/*.wav

    Lo stesso file fisico viene copiato (come hardlink se possibile,
    altrimenti copiato) sotto le 3 viste, perche' pyAudioAnalysis
    organizza i dati per cartella-di-classe e non supporta multi-label
    nativamente. Ritorna anche la lista di metadati effettivamente usati,
    utile per la valutazione successiva.
    """
    workspace_dir = Path(workspace_dir)
    if workspace_dir.exists():
        shutil.rmtree(workspace_dir)

    for slot in SLOTS:
        (workspace_dir / slot).mkdir(parents=True, exist_ok=True)

    used_metadata = []

    for m in audio_metadata:
        if m["speaker_id"] not in speaker_ids:
            continue

        src = resolve_audio_path(dataset_path, m["rel_path"])
        if src is None:
            continue

        # Nome file univoco anche se piu' speaker avessero file omonimi
        dest_name = f"{m['speaker_id']}__{src.name}"

        for slot in SLOTS:
            class_value = m[slot]
            class_dir = workspace_dir / slot / class_value
            class_dir.mkdir(parents=True, exist_ok=True)
            dest = class_dir / dest_name
            if not dest.exists():
                try:
                    os.link(src, dest)
                except OSError:
                    shutil.copy(src, dest)

        # Percorso fisico di riferimento (qualsiasi slot va bene: il file
        # e' identico nelle 3 viste, cambia solo la cartella di classe)
        workspace_path = workspace_dir / "object" / m["object"] / dest_name
        used_metadata.append({**m, "_dest_name": dest_name, "_workspace_path": str(workspace_path)})

    return used_metadata


def train_slot_models(workspace_dir, model_type, model_prefix):
    """
    Allena un classificatore (model_type: "svm_rbf" oppure "gradientboosting")
    per ciascuno dei 3 slot (action/object/location), usando le sotto-cartelle
    per classe create da `build_split_workspace`.

    Ritorna un dict {slot: model_name} con i nomi dei modelli salvati.
    """
    workspace_dir = Path(workspace_dir)
    model_names = {}

    for slot in SLOTS:
        slot_dir = workspace_dir / slot
        class_folders = sorted(str(p) for p in slot_dir.iterdir() if p.is_dir())

        if len(class_folders) < 2:
            print(f"Skip training per slot '{slot}': servono almeno 2 classi, trovate {len(class_folders)}.")
            continue

        model_name = f"{model_prefix}_{slot}"
        print(f"\nTraining {model_type} per slot '{slot}' ({len(class_folders)} classi)...")

        aT.extract_features_and_train(
            class_folders,
            1.0, 1.0,   # mt win/step
            0.05, 0.025,   # st win/step
            model_type,
            model_name,
            False
        )

        model_names[slot] = model_name
        print(f"Training completato: {model_name}")

    return model_names

In [9]:
def predict_intent_json(wav_path, model_names, model_type):
    """
    Predice i 3 slot (action, object, location) per un singolo file audio
    usando i modelli (uno per slot) indicati in `model_names`, e li
    combina in un unico oggetto json:

        {"object": ..., "action": ..., "location": ...}

    Ritorna (intent_json, raw_pred_dict) dove raw_pred_dict contiene la
    predizione "grezza" per ogni slot (utile per calcolare le accuracy
    per singolo slot).
    """
    raw_pred = {}

    for slot, model_name in model_names.items():
        try:
            class_id, class_names, _ = aT.file_classification(str(wav_path), model_name, model_type)
            pred_label = class_names[int(class_id)]
        except Exception as e:
            pred_label = None
        raw_pred[slot] = pred_label

    intent_json = {
        "object": raw_pred.get("object"),
        "action": raw_pred.get("action"),
        "location": raw_pred.get("location"),
    }

    return intent_json, raw_pred


def evaluate_split(used_metadata, model_names, model_type):
    """
    Valuta un insieme di campioni (test) con i modelli allenati
    per i 3 slot. Per ogni file produce il json dell'intento predetto e lo
    confronta con ground truth.

    Ritorna:
        - results: lista di dict con true/pred per ogni file (incluso il
          json completo predetto)
        - metrics: dict con accuracy per slot + exact match accuracy
          (il json predetto deve coincidere ESATTAMENTE su tutti e 3 i
          campi con quello reale)
    """
    results = []
    y_true_slot = {slot: [] for slot in SLOTS}
    y_pred_slot = {slot: [] for slot in SLOTS}
    exact_matches = 0

    for m in used_metadata:
        wav_path = Path(m["_workspace_path"])

        pred_json, raw_pred = predict_intent_json(wav_path, model_names, model_type)

        true_json = {
            "object": m["object"],
            "action": m["action"],
            "location": m["location"],
        }

        for slot in SLOTS:
            y_true_slot[slot].append(m[slot])
            y_pred_slot[slot].append(raw_pred.get(slot))

        is_exact_match = (pred_json == true_json)
        exact_matches += int(is_exact_match)

        results.append({
            "rel_path": m["rel_path"],
            "speaker_id": m["speaker_id"],
            "true_intent": true_json,
            "pred_intent": pred_json,
            "exact_match": is_exact_match,
        })

    n = len(used_metadata)
    metrics = {
        "n_samples": n,
        "exact_match_accuracy": exact_matches / n if n else 0.0,
        "slot_accuracy": {},
    }

    for slot in SLOTS:
        yt = y_true_slot[slot]
        yp = y_pred_slot[slot]
        correct = sum(1 for t, p in zip(yt, yp) if t == p)
        metrics["slot_accuracy"][slot] = correct / n if n else 0.0

    return results, metrics

In [10]:
def plot_slot_confusion_matrices(results, slot, title_prefix):
    """
    Disegna la confusion matrix per un singolo slot (action/object/location),
    usando le etichette osservate in `results` (true_intent / pred_intent).
    """
    plots_dir = Path("./plots")
    plots_dir.mkdir(exist_ok=True)

    def normalize_label(v):
        # Unifica None/NaN in una singola etichetta e converte il resto in stringa
        return "<MISSING>" if pd.isna(v) else str(v)

    y_true = [normalize_label(r["true_intent"][slot]) for r in results]
    y_pred = [normalize_label(r["pred_intent"][slot]) for r in results]

    labels = sorted(set(y_true) | set(y_pred))
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    plt.figure(figsize=(max(5, len(labels) * 0.6), max(4, len(labels) * 0.6)))
    plt.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.title(f"{title_prefix} - {slot}", weight="bold")
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(range(len(labels)), labels, rotation=90)
    plt.yticks(range(len(labels)), labels)

    thresh = cm.max() / 2 if cm.max() > 0 else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black",
                     fontsize=8)

    plt.tight_layout()
    clean_name = f"{title_prefix}_{slot}".lower().replace(" ", "_")
    plot_path = plots_dir / f"confusion_matrix_{clean_name}.png"
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print(f"Grafico salvato in: {plot_path}")

In [11]:
def print_sample_predictions(results, n=5, title="Esempi di predizione"):
    """
    Stampa alcuni esempi di output: per ciascun file mostra il json
    dell'intento reale e quello predetto, nel formato richiesto
    {"object": ..., "action": ..., "location": ...}.
    """
    print(f"\n--- {title} (primi {n} esempi) ---")
    for r in results[:n]:
        print(f"file: {r['rel_path']}")
        print(f"  true: {json.dumps(r['true_intent'])}")
        print(f"  pred: {json.dumps(r['pred_intent'])}  (exact_match={r['exact_match']})")


In [12]:
def plot_summary(svm_metrics, gbc_metrics, split_name):
    """
    Confronta SVM vs GradientBoosting su un determinato split (dev/test):
    accuracy per slot + exact match accuracy, in un unico grafico a barre.
    """
    plots_dir = Path("./plots")
    plots_dir.mkdir(exist_ok=True)

    labels = SLOTS + ["exact_match"]
    svm_vals = [svm_metrics["slot_accuracy"][s] for s in SLOTS] + [svm_metrics["exact_match_accuracy"]]
    gbc_vals = [gbc_metrics["slot_accuracy"][s] for s in SLOTS] + [gbc_metrics["exact_match_accuracy"]]

    x = np.arange(len(labels))
    width = 0.35

    plt.figure(figsize=(8, 5))
    bars_svm = plt.bar(x - width/2, svm_vals, width, label='SVM', color='#34495e')
    bars_gbc = plt.bar(x + width/2, gbc_vals, width, label='GradientBoosting', color='#1abc9c')

    plt.ylabel('Accuracy')
    plt.title(f'SVM vs GradientBoosting - {split_name}', weight='bold')
    plt.xticks(x, labels)
    plt.ylim(0, 1.1)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.legend()

    for bars in [bars_svm, bars_gbc]:
        for bar in bars:
            yval = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.02,
                     f"{yval:.2%}", ha='center', va='bottom', weight='bold', fontsize=9)

    plt.tight_layout()
    summary_path = plots_dir / f"accuracy_comparison_{split_name.lower()}.png"
    plt.savefig(summary_path, dpi=150)
    plt.show()
    print(f"Grafico riassuntivo salvato in: {summary_path}")

In [13]:
# Diagnostic: find out where the .wav files actually live on disk,
# since resolve_audio_path's guessed candidates may not match the
# real kagglehub download layout. Run this once and inspect the output
# before re-running the training/eval loop below.

print("Top-level contents of dataset_path:")
for p in sorted(dataset_path.iterdir()):
    print(" ", p.name, "[dir]" if p.is_dir() else "[file]")

print()
print("First 5 .wav files found anywhere under dataset_path:")
count = 0
for wav in dataset_path.rglob("*.wav"):
    print(" ", wav.relative_to(dataset_path))
    count += 1
    if count >= 5:
        break

if count == 0:
    print("  No .wav files found at all under dataset_path!")

Top-level contents of dataset_path:
  fluent_speech_commands_dataset [dir]

First 5 .wav files found anywhere under dataset_path:
  fluent_speech_commands_dataset/wavs/speakers/9MX3AgZzVgCw4W4j/a7d22f30-450a-11e9-9539-7f047cfe24d1.wav
  fluent_speech_commands_dataset/wavs/speakers/9MX3AgZzVgCw4W4j/44cfebd0-4509-11e9-9539-7f047cfe24d1.wav
  fluent_speech_commands_dataset/wavs/speakers/9MX3AgZzVgCw4W4j/33f34bd0-4500-11e9-a1ea-79ca03012c0e.wav
  fluent_speech_commands_dataset/wavs/speakers/9MX3AgZzVgCw4W4j/1b4f5aa0-4501-11e9-a1ea-79ca03012c0e.wav
  fluent_speech_commands_dataset/wavs/speakers/9MX3AgZzVgCw4W4j/ef1326b0-450a-11e9-9539-7f047cfe24d1.wav


In [14]:
audio_metadata = load_complete_map(dataset_path)

# --- 1. Costruzione split a livello di SPEAKER (richiesta 3 e 4) ---
# train/dev: solo uomini, eta 22-40
# test:      tutti gli altri speaker
# nessuno speaker e' condiviso tra (train+dev) e test
train_speakers, young_female, middle_age_female, old_female, middle_age_male, old_male = build_speaker_splits(audio_metadata)

# --- 2. Costruzione dei workspace fisici (cartelle per classe per slot) ---
train_meta = build_split_workspace(audio_metadata, dataset_path, train_speakers, "./workspace_train")
young_female_test_meta  = build_split_workspace(audio_metadata, dataset_path, young_female,  "./workspace_test/young_female")
middle_age_female_test_meta  = build_split_workspace(audio_metadata, dataset_path, middle_age_female,  "./workspace_test/middle_age_female")
old_female_test_meta  = build_split_workspace(audio_metadata, dataset_path, old_female,  "./workspace_test/old_female")
middle_age_male_test_meta  = build_split_workspace(audio_metadata, dataset_path, middle_age_male,  "./workspace_test/middle_age_male")
old_male_test_meta  = build_split_workspace(audio_metadata, dataset_path, old_male,  "./workspace_test/old_male")

print(f"Campioni train: {len(train_meta)}")
print(f"Campioni test donne giovani:  {len(young_female_test_meta)}")
print(f"Campioni test donne mezza età:  {len(middle_age_female_test_meta)}")
print(f"Campioni test donne vecchie:  {len(old_female_test_meta)}")
print(f"Campioni test uomini mezza età:  {len(middle_age_male_test_meta)}")
print(f"Campioni test uomini vecchi:  {len(old_male_test_meta)}")


print("OK: nessuno speaker di train e' presente in test (e viceversa).")

Loaded metadata: 30043 samples
Speaker totali: 97
  -> train (M, 22-40): 40 speaker
  -> test: donne giovani: 35 speaker
  -> test: donne a 41-65: 11 speaker
  -> test: donne a 65+: 1 speaker
  -> test: uomini a 41-65: 8 speaker
  -> test: uomini a 65+: 2 speaker
Campioni train: 12156
Campioni test donne giovani:  10168
Campioni test donne mezza età:  3866
Campioni test donne vecchie:  137
Campioni test uomini mezza età:  3232
Campioni test uomini vecchi:  484
OK: nessuno speaker di train e' presente in test (e viceversa).


In [4]:
svm_action = aT.load_model("model_intent_svm_action", False)
svm_object = aT.load_model("model_intent_svm_object", False)
svm_location = aT.load_model("model_intent_svm_location", False)
gbc_action = aT.load_model("model_intent_gbc_action", False)
gbc_object = aT.load_model("model_intent_gbc_object", False)
gbc_location = aT.load_model("model_intent_gbc_location", False)

In [5]:
aT.file_classification("/home/Cerviniadmin/.cache/kagglehub/datasets/tommyngx/fluent-speech-corpus/versions/1/fluent_speech_commands_dataset/wavs/speakers/2BqVo8kVB2Skwgyb/0a3129c0-4474-11e9-a9a5-5dbec3b8816a.wav", "model_intent_svm_action","svm_rbf")

(np.float64(2.0),
 array([1.69436855e-02, 1.64859942e-03, 9.80443376e-01, 3.79735246e-04,
        2.17909467e-04, 3.66694681e-04]),
 ['activate',
  'bring',
  'change language',
  'deactivate',
  'decrease',
  'increase'])

In [16]:
svm_models = {
    "action": "model_intent_svm_action",
    "object": "model_intent_svm_object",
    "location": "model_intent_svm_location"
}
gbc_models = {
    "action": "model_intent_gbc_action",
    "object": "model_intent_gbc_object",
    "location": "model_intent_gbc_location"
}

In [ ]:
test_splits = [
    ("Test donne giovani", young_female_test_meta),
    ("Test donne mezza eta", middle_age_female_test_meta),
    ("Test donne vecchie", old_female_test_meta),
    ("Test uomini mezza eta", middle_age_male_test_meta),
    ("Test uomini vecchi", old_male_test_meta),
]

all_metrics = {}

for split_name, split_meta in test_splits:
    print(f"\n=== Valutazione su {split_name} ===")

    svm_results, svm_metrics = evaluate_split(split_meta, svm_models, "svm_rbf")
    gbc_results, gbc_metrics = evaluate_split(split_meta, gbc_models, "gradientboosting")

    print(f"[SVM] {split_name} - exact match accuracy: {svm_metrics['exact_match_accuracy']:.2%}")
    print(f"[SVM] {split_name} - accuracy per slot: {svm_metrics['slot_accuracy']}")
    print(f"[GBC] {split_name} - exact match accuracy: {gbc_metrics['exact_match_accuracy']:.2%}")
    print(f"[GBC] {split_name} - accuracy per slot: {gbc_metrics['slot_accuracy']}")

    print_sample_predictions(svm_results, n=5, title=f"SVM - {split_name}")
    print_sample_predictions(gbc_results, n=5, title=f"GBC - {split_name}")

    for slot in SLOTS:
        plot_slot_confusion_matrices(svm_results, slot, f"SVM {split_name}")
        plot_slot_confusion_matrices(gbc_results, slot, f"GBC {split_name}")

    plot_summary(svm_metrics, gbc_metrics, split_name)
    all_metrics[split_name] = {"svm": svm_metrics, "gbc": gbc_metrics}

print("\nRiepilogo finale metriche:")
print(json.dumps(all_metrics, indent=2))


=== Valutazione su Test donne giovani ===


/home/Cerviniadmin/.local/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/Cerviniadmin/.local/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/Cerviniadmin/.local/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.9.0 when using version 1.7.2. This might lead to break